In [1]:
import nltk
import re
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from collections import Counter
nltk.download('punkt')
nltk.download('stopwords')
import pandas as pd
from numpy.linalg import norm
from scipy.stats import entropy, shapiro, mannwhitneyu
from nltk.util import ngrams

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\shant\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\shant\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [2]:
import nltk
from nltk.stem import PorterStemmer

nltk.download('punkt')  # Download the Punkt tokenizer if you haven't already
nltk.download('averaged_perceptron_tagger')  # Download POS tagger data (needed for lemmatization)
nltk.download('wordnet')  # Download WordNet data (needed for lemmatization)

stemmer = PorterStemmer()

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\shant\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     C:\Users\shant\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\shant\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [3]:
def preprocess_and_tokenize(text, n_grams):
    # Remove URLs, special characters, and numbers
    text = re.sub(r'http\S+|www\S+|https\S+|\d+|\W+', ' ', text).lower()
    
    # Tokenize the text
    tokens = word_tokenize(text)
    
    # Remove stopwords
    stop_words = set(stopwords.words('english'))
    tokens = [word for word in tokens if word not in stop_words]
    
    # Stemming
    tokens = [stemmer.stem(word) for word in tokens]
    n_grams = list(ngrams(tokens,n_grams))
    return n_grams

In [4]:
def preprocess_dataset(dataset, n_grams, prune=True,high_prune=500,low_prune=750):
    words = [] 
    for data in dataset : 
        tokens = preprocess_and_tokenize(data, n_grams)
        words.extend(tokens)
    if prune:
        mc = Counter(words).most_common()
        vocab_prune=[]
        for word,count in mc[:high_prune]:
            vocab_prune.extend([word]*count)
        for word,count in mc[-low_prune:]:
            vocab_prune.extend([word]*count)
        return vocab_prune
    else:
        return words

In [5]:
def JSD(P, Q):
    _P = P / norm(P, ord=1)
    _Q = Q / norm(Q, ord=1)
    _M = 0.5 * (_P + _Q)
    return 0.5 * (entropy(_P, _M) + entropy(_Q, _M))

In [6]:
def binary_compare(df, n_grams,multi=True,prune=False):
    #df = pd.read_csv(BASE)
    num_unq = len(df["label"].unique())
    column = 'text'
    dataset_1 = preprocess_dataset(df[df['label'] ==1][column].values.tolist(),prune=prune, n_grams = n_grams)
    print("dataset_1 size: ", len(dataset_1))
    dataset_0 = preprocess_dataset(df[df['label'] == 0][column].values.tolist(),prune=prune, n_grams = n_grams)
    print("dataset_0 size: ", len(dataset_0))

    set_1 = set(dataset_1)
    print("set_1 size: ", len(set_1))
    set_0 = set(dataset_0)
    print("set_0 size: ", len(set_0))
    union_set = set_0.union(set_1)
    print("union_set size: ", len(union_set))

    counter_1 = Counter(dataset_1)
    counter_0 = Counter(dataset_0)

    pdf_1 = []
    pdf_0 = []
    pdf_1_raw = []
    pdf_0_raw = []
    N = len(dataset_1) + len(dataset_0)
    V = len(union_set)

    for word in union_set : 
        if word in set_1 : 
            pdf_1.append((counter_1[word]+1)/(len(dataset_1) + len(set_1)))
        else : 
            pdf_1.append(1/(N+V))
        pdf_1_raw.append(counter_1.get(word,0))
        if word in set_0 : 
            pdf_0.append((counter_0[word]+1)/(len(dataset_0) + len(set_0)))
        else : 
            pdf_0.append(1/(N+V))
        pdf_0_raw.append(counter_0.get(word,0)) 
    print(sum(pdf_1))
    print(sum(pdf_0))

    print("\n\n ---- JS")
    print("Jenson Shannon Distance for 1 0: ", round(JSD(pdf_1, pdf_0),4))

In [7]:
df = pd.read_csv(r'D:\Projects and Publications\Mental Health Comorbidity Detection\Git\mental-health-comorbitidy-classification\data\test/full_test.csv')

In [8]:
df.head()

,id,title,selftext,multilabel_clf_label,disorder,text,multiclass_clf_label,depression_label,anxiety_label
0,8es6qn,I cut myself for the first time in a year toda...,... and hated that I still loved it. The burni...,"[1, 0]",{'depressive_disorder'},I cut myself for the first time in a year toda...,Depression,1,0
1,ar89pm,Anybody hate spring/summer?,".....And even so, when depression is not too s...","[1, 1]","{'depressive_disorder', 'anxiety_disorder'}","Anybody hate springsummer. And even so, when d...",Comorbid (Depression + Anxiety),1,1
2,5g4v1n,So I'm staring at this tablet of Lexapro...,...and I'm not sure what to do. I guess I don'...,"[1, 1]","{'depressive_disorder', 'anxiety_disorder'}",So Im staring at this tablet of Lexapro. and I...,Comorbid (Depression + Anxiety),1,1
3,9x8h3g,"Found a poem sort of thing, very emo (ha ha ha...","""I am delicate and bitter. I am sweet on the o...","[1, 0]",{'depressive_disorder'},"Found a poem sort of thing, very emo ha ha ha ...",Depression,1,0
4,es440i,You don't get it,"""It gets better"" ""Stop thinking about it"" ""get...","[1, 1]","{'depressive_disorder', 'anxiety_disorder'}",You dont get it. It gets better Stop thinking ...,Comorbid (Depression + Anxiety),1,1


### MHCA

In [9]:
df_control = df[(df['anxiety_label'] == 0) & (df['depression_label'] == 0)]
df_anxiety_comorbidity = df[df['anxiety_label'] == 1]
df_depression_comorbidity = df[df['depression_label'] == 1]
df_comorbidity = df[(df['anxiety_label'] == 1) & (df['depression_label'] == 1)]

In [10]:
dfs = [df_control, df_anxiety_comorbidity, df_depression_comorbidity, df_comorbidity]
dfs_names = ['df_control', 'df_anxiety_comorbidity', 'df_depression_comorbidity', 'df_comorbidity']

In [11]:
n_grams = 1
for i in range(len(dfs)):
    for j in range(i,len(dfs)):
        df1 = dfs[i]
        df2 = dfs[j]
        print(dfs_names[i], dfs_names[j])
        df_new = pd.DataFrame({'text': df1['text'].tolist() + df2['text'].tolist(), 'label': [0]*len(df1) + [1]*len(df2)})
        binary_compare(df_new, n_grams = 1)
        print('_____________________________________________________________________')

df_control df_control
dataset_1 size:  110869
dataset_0 size:  110869
set_1 size:  6352
set_0 size:  6352
union_set size:  6352
1.000000000000054
1.000000000000054


 ---- JS
Jenson Shannon Distance for 1 0:  0.0
_____________________________________________________________________
df_control df_anxiety_comorbidity
dataset_1 size:  71103
dataset_0 size:  110869
set_1 size:  4837
set_0 size:  6352
union_set size:  7785
1.015535658763605
1.0075517635713618


 ---- JS
Jenson Shannon Distance for 1 0:  0.0336
_____________________________________________________________________
df_control df_depression_comorbidity
dataset_1 size:  154770
dataset_0 size:  110869
set_1 size:  6766
set_0 size:  6352
union_set size:  9017
1.0081957066293734
1.0097030467203139


 ---- JS
Jenson Shannon Distance for 1 0:  0.0273
_____________________________________________________________________
df_control df_comorbidity
dataset_1 size:  61812
dataset_0 size:  110869
set_1 size:  4471
set_0 size:  6352
union_s

In [12]:
n_grams = 1
for i in range(len(dfs)):
    for j in range(i,len(dfs)):
        df1 = dfs[i]
        df2 = dfs[j]
        print(dfs_names[i], dfs_names[j])
        df_new = pd.DataFrame({'text': df1['text'].tolist() + df2['text'].tolist(), 'label': [0]*len(df1) + [1]*len(df2)})
        binary_compare(df_new, n_grams = 2)
        print('_____________________________________________________________________')

df_control df_control
dataset_1 size:  109668
dataset_0 size:  109668
set_1 size:  72863
set_0 size:  72863
union_set size:  72863
0.9999999999986076
0.9999999999986076


 ---- JS
Jenson Shannon Distance for 1 0:  0.0
_____________________________________________________________________
df_control df_anxiety_comorbidity
dataset_1 size:  70402
dataset_0 size:  109668
set_1 size:  47619
set_0 size:  72863
union_set size:  106672
1.205944716853752
1.1179073871273717


 ---- JS
Jenson Shannon Distance for 1 0:  0.1497
_____________________________________________________________________
df_control df_depression_comorbidity
dataset_1 size:  153191
dataset_0 size:  109668
set_1 size:  88923
set_0 size:  72863
union_set size:  140891
1.1287133126931517
1.1684904024750986


 ---- JS
Jenson Shannon Distance for 1 0:  0.1416
_____________________________________________________________________
df_control df_comorbidity
dataset_1 size:  61203
dataset_0 size:  109668
set_1 size:  41900
set_0 size:

In [13]:
n_grams = 1
for i in range(len(dfs)):
    for j in range(i,len(dfs)):
        df1 = dfs[i]
        df2 = dfs[j]
        print(dfs_names[i], dfs_names[j])
        df_new = pd.DataFrame({'text': df1['text'].tolist() + df2['text'].tolist(), 'label': [0]*len(df1) + [1]*len(df2)})
        binary_compare(df_new, n_grams = 3)
        print('_____________________________________________________________________')

df_control df_control
dataset_1 size:  108467
dataset_0 size:  108467
set_1 size:  103221
set_0 size:  103221
union_set size:  103221
0.9999999999989644
0.9999999999989644


 ---- JS
Jenson Shannon Distance for 1 0:  0.0
_____________________________________________________________________
df_control df_anxiety_comorbidity
dataset_1 size:  69701
dataset_0 size:  108467
set_1 size:  65913
set_0 size:  103221
union_set size:  164603
1.2879181727736058
1.1790758261331198


 ---- JS
Jenson Shannon Distance for 1 0:  0.1874
_____________________________________________________________________
df_control df_depression_comorbidity
dataset_1 size:  151612
dataset_0 size:  108467
set_1 size:  140182
set_0 size:  103221
union_set size:  236139
1.1933767013685463
1.2678621089901216


 ---- JS
Jenson Shannon Distance for 1 0:  0.1846
_____________________________________________________________________
df_control df_comorbidity
dataset_1 size:  60594
dataset_0 size:  108467
set_1 size:  57414
set_

### Depression Reddit

In [16]:
import os

# Create an empty DataFrame
df = pd.DataFrame(columns=["text", "label"])

folder_d = r'D:\Projects and Publications\Mental Health Comorbidity Detection\Git\mental-health-comorbitidy-classification\data\existing_datasets\Depression_Reddit\mixed_depression'
folder_nd = r'D:\Projects and Publications\Mental Health Comorbidity Detection\Git\mental-health-comorbitidy-classification\data\existing_datasets\Depression_Reddit\mixed_non_depression'

folders = [folder_nd, folder_d]

encodings_to_try = ["utf-8", "latin-1", "utf-16", "utf-32", "cp1252"]

# Loop through the folders and read text files with different encodings
for label, folder in enumerate(folders):
    for filename in os.listdir(folder):
        if filename.endswith(".txt"):
            file_path = os.path.join(folder, filename)
            for encoding in encodings_to_try:
                try:
                    with open(file_path, 'r', encoding=encoding) as file:
                        text = file.read()
                        df = pd.concat([df, pd.DataFrame({"text": [text], "label": [label]})], ignore_index=True)
                    break  # If successful, break out of the encoding loop
                except UnicodeDecodeError:
                    pass  # Try the next encoding


df["label"] = df["label"].astype(int)

In [20]:
binary_compare(df, n_grams = 1)

dataset_1 size:  128876
dataset_0 size:  147999
set_1 size:  6987
set_0 size:  10385
union_set size:  12636
1.0195122119713405
1.0077751795268353


 ---- JS
Jenson Shannon Distance for 1 0:  0.0604


In [21]:
binary_compare(df, n_grams = 2)

dataset_1 size:  127536
dataset_0 size:  146519
set_1 size:  87086
set_0 size:  106501
union_set size:  175791
1.1971897049196178
1.1540304904331826


 ---- JS
Jenson Shannon Distance for 1 0:  0.1648


### DATD

In [20]:
df = pd.read_csv(r'D:\Projects and Publications\Mental Health Comorbidity Detection\Git\mental-health-comorbitidy-classification\data\existing_datasets\DATD/DATD_training.csv')
label = []
for i in df['label']:
    if i == 'MENTAL_HEALTH':
        label.append(1)
    else:
        label.append(0)
df = pd.DataFrame({'text':df['text'].tolist(), 'label':label})
binary_compare(df, n_grams = 1)

dataset_1 size:  3142
dataset_0 size:  3224
set_1 size:  1055
set_0 size:  1341
union_set size:  1881
1.1001576330787046
1.0654783557657492


 ---- JS
Jenson Shannon Distance for 1 0:  0.1039


In [22]:
df.to_csv('DATD.csv', index = False)

In [23]:
df = pd.read_csv(r'D:\Projects and Publications\Mental Health Comorbidity Detection\Git\mental-health-comorbitidy-classification\data\existing_datasets\DATD/DATD_training.csv')
label = []
for i in df['label']:
    if i == 'MENTAL_HEALTH':
        label.append(1)
    else:
        label.append(0)
df = pd.DataFrame({'text':df['text'].tolist(), 'label':label})
binary_compare(df, n_grams = 2)

dataset_1 size:  2669
dataset_0 size:  2797
set_1 size:  2363
set_0 size:  2554
union_set size:  4737
1.2326766637264346
1.2139566794080883


 ---- JS
Jenson Shannon Distance for 1 0:  0.1817


# dreddit

1: stress

0: non stress

In [24]:
dreddit = r"D:\Projects and Publications\Mental Health Comorbidity Detection\Git\mental-health-comorbitidy-classification\data\existing_datasets\Dreaddit\dreaddit-train.csv"
df = pd.read_csv(dreddit)
binary_compare(df, n_grams = 1)

dataset_1 size:  57723
dataset_0 size:  50592
set_1 size:  5075
set_0 size:  5527
union_set size:  7313
1.0193551734873252
1.0154460857231633


 ---- JS
Jenson Shannon Distance for 1 0:  0.0555


In [26]:
df[['text', 'label']].to_csv('dreddit.csv', index = False)

In [26]:
dreddit = r"D:\Projects and Publications\Mental Health Comorbidity Detection\Git\mental-health-comorbitidy-classification\data\existing_datasets\Dreaddit\dreaddit-train.csv"
df = pd.read_csv(dreddit)
binary_compare(df, n_grams = 2)

dataset_1 size:  56235
dataset_0 size:  49242
set_1 size:  44717
set_0 size:  40621
union_set size:  79488
1.1879869164435422
1.2101316465275562


 ---- JS
Jenson Shannon Distance for 1 0:  0.1736


### SDCNL

In [27]:
df = pd.read_csv(r'D:\Projects and Publications\Mental Health Comorbidity Detection\Git\mental-health-comorbitidy-classification\data\existing_datasets\SDCNL/combined-set.csv')
df = pd.DataFrame({'text': df['selftext_clean'].tolist(), 'label': df['is_suicide'].tolist()})
df = df.dropna()
binary_compare(df, n_grams = 1)

dataset_1 size:  70618
dataset_0 size:  77558
set_1 size:  5168
set_0 size:  5431
union_set size:  7204
1.013103359505788
1.0114107349723453


 ---- JS
Jenson Shannon Distance for 1 0:  0.031


In [29]:
df.to_csv("SDCNL.csv", index = False)

In [29]:
df = pd.read_csv(r'D:\Projects and Publications\Mental Health Comorbidity Detection\Git\mental-health-comorbitidy-classification\data\existing_datasets\SDCNL/combined-set.csv')
df = pd.DataFrame({'text': df['selftext_clean'].tolist(), 'label': df['is_suicide'].tolist()})
df = df.dropna()
binary_compare(df, n_grams = 2)

dataset_1 size:  69639
dataset_0 size:  76643
set_1 size:  50009
set_0 size:  54910
union_set size:  93366
1.1809195152899217
1.160468687408781


 ---- JS
Jenson Shannon Distance for 1 0:  0.1503
